In [ ]:
!pip install transformers
# pip installs Python libraries.
# transformers is Hugging Face's Python library.
# It lets us load and use pretrained models.

In [ ]:
from transformers import pipeline

In [ ]:
sentiment_model = pipeline("sentiment-analysis")
# pipeline() is a Hugging Face function.
# "sentiment-analysis" tells it which task we want.
# Hugging Face automatically selects a suitable pretrained model if we don't specify one.
# sentiment_model stores the pipeline so we can use it later.

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [ ]:
review = "This movie was amazing. I really enjoyed it!"
result = sentiment_model(review)
print(result)

[{'label': 'POSITIVE', 'score': 0.9998794794082642}]


In [ ]:
reviews = [
    "This movie was terrible. I hated it.",
    "The movie was okay, nothing special.",
    "The acting was brilliant and the story was beautiful."
]

for review in reviews:
    print(review)
    print(sentiment_model(review))
    print()

This movie was terrible. I hated it.
[{'label': 'NEGATIVE', 'score': 0.9997337460517883}]

The movie was okay, nothing special.
[{'label': 'NEGATIVE', 'score': 0.9925116300582886}]

The acting was brilliant and the story was beautiful.
[{'label': 'POSITIVE', 'score': 0.999887228012085}]



In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
text = "I love this movie"

tokens = tokenizer(text)

print(tokens)

{'input_ids': [101, 1045, 2293, 2023, 3185, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1]}


In [ ]:
print(tokenizer.tokenize("I love this movie"))

print(tokenizer.convert_tokens_to_ids(
    tokenizer.tokenize("I love this movie")
))

['i', 'love', 'this', 'movie']
[1045, 2293, 2023, 3185]


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_name)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
text = "I love this movie"

inputs = tokenizer(text, return_tensors="pt")

print(inputs)

{'input_ids': tensor([[ 101, 1045, 2293, 2023, 3185,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}


In [ ]:
outputs = model(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"]
)

print(outputs.logits)

tensor([[-4.3237,  4.6761]], grad_fn=<AddmmBackward0>)


In [ ]:
import torch

probabilities = torch.softmax(outputs.logits, dim=1)

print(probabilities)

tensor([[1.2341e-04, 9.9988e-01]], grad_fn=<SoftmaxBackward0>)


In [ ]:
predicted_class_id = outputs.logits.argmax(dim=1).item()

print("Class ID:", predicted_class_id)
print("Sentiment:", model.config.id2label[predicted_class_id])

Class ID: 1
Sentiment: POSITIVE


In [ ]:
reviews = [
    "This movie was fantastic!",
    "I hated this movie.",
    "The acting was amazing.",
    "The story was boring."
]

results = sentiment_model(reviews)

print(results)

[{'label': 'POSITIVE', 'score': 0.9998781681060791}, {'label': 'NEGATIVE', 'score': 0.9997289776802063}, {'label': 'POSITIVE', 'score': 0.9998806715011597}, {'label': 'NEGATIVE', 'score': 0.9997658133506775}]


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "Review": reviews,
    "Sentiment": [result["label"] for result in results],
    "Confidence": [result["score"] for result in results]
})

df

,Review,Sentiment,Confidence
0,This movie was fantastic!,POSITIVE,0.999878
1,I hated this movie.,NEGATIVE,0.999729
2,The acting was amazing.,POSITIVE,0.999881
3,The story was boring.,NEGATIVE,0.999766


In [ ]:
def analyze_reviews(reviews):
    results = sentiment_model(reviews)

    df = pd.DataFrame({
        "Review": reviews,
        "Sentiment": [r["label"] for r in results],
        "Confidence": [round(r["score"] * 100, 2) for r in results]
    })

    return df

In [ ]:
new_reviews = [
    "I really enjoyed this movie!",
    "The movie was a complete waste of time.",
    "It was an average movie."
]

analyze_reviews(new_reviews)

,Review,Sentiment,Confidence
0,I really enjoyed this movie!,POSITIVE,99.99
1,The movie was a complete waste of time.,NEGATIVE,99.98
2,It was an average movie.,NEGATIVE,68.32


In [ ]:
test_reviews = [
    "The movie was neither good nor bad.",
    "It was an average movie.",
    "The movie was okay.",
    "Absolutely loved this masterpiece!"
]

analyze_reviews(test_reviews)

,Review,Sentiment,Confidence
0,The movie was neither good nor bad.,NEGATIVE,99.38
1,It was an average movie.,NEGATIVE,68.32
2,The movie was okay.,POSITIVE,99.98
3,Absolutely loved this masterpiece!,POSITIVE,99.99
